# Индивидуальный проект: А/В-тестирование

In [1]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize, proportions_ztest

In [2]:
base = Path("../../misc/ab_individual_project")

opened = pd.read_csv(base / "Data for MDE - opened.csv")
clicked = pd.read_csv(base / "Data for MDE - clicked.csv")
monitoring = pd.read_csv(base / "Data for monitoring - first 4 days.csv")
results = pd.read_csv(base / "Data for results.csv")

opened["date"] = pd.to_datetime(opened["date"], dayfirst=True)
clicked["date"] = pd.to_datetime(clicked["date"], dayfirst=True)
monitoring["date"] = pd.to_datetime(monitoring["date"], dayfirst=True)
results["date"] = pd.to_datetime(results["date"], dayfirst=True)

opened.shape, clicked.shape, monitoring.shape, results.shape

((4853, 2), (490, 2), (966, 4), (1205, 4))

## Задание 1. Приоритизация гипотез (ICE и RICE)

### Исходные оценки гипотез
- 1) Яркие заголовки email: Impact=4, Reach=8, Confidence=4, Ease=8
- 2) Улучшение mobile-версии: Impact=8, Reach=6, Confidence=6, Ease=2
- 3) Персонализированные email: Impact=5, Reach=6, Confidence=4, Ease=5
- 4) Интерактивные email: Impact=2, Reach=4, Confidence=4, Ease=7
- 5) Рекомендации в ЛК: Impact=9, Reach=2, Confidence=7, Ease=3

### Формулы
- ICE = Impact × Confidence × Ease
- Для RICE требуется Effort. В данных дана простота (Ease), поэтому примем `Effort = 11 - Ease`.
- RICE = Reach × Impact × Confidence / Effort

In [3]:
hyp = pd.DataFrame(
    [
        ["Яркие заголовки email", 4, 8, 4, 8],
        ["Улучшение mobile-версии", 8, 6, 6, 2],
        ["Персонализированные email", 5, 6, 4, 5],
        ["Интерактивные email", 2, 4, 4, 7],
        ["Рекомендации в ЛК", 9, 2, 7, 3],
    ],
    columns=["hypothesis", "impact", "reach", "confidence", "ease"],
)

hyp["ice"] = hyp["impact"] * hyp["confidence"] * hyp["ease"]
hyp["effort"] = 11 - hyp["ease"]
hyp["rice"] = hyp["reach"] * hyp["impact"] * hyp["confidence"] / hyp["effort"]

hyp.sort_values("rice", ascending=False)

,hypothesis,impact,reach,confidence,ease,ice,effort,rice
0,Яркие заголовки email,4,8,4,8,128,3,42.666667
1,Улучшение mobile-версии,8,6,6,2,96,9,32.000000
2,Персонализированные email,5,6,4,5,100,6,20.000000
4,Рекомендации в ЛК,9,2,7,3,189,8,15.750000
3,Интерактивные email,2,4,4,7,56,4,8.000000


### Вывод по заданию 1
Для запуска в ближайшую неделю выбираю гипотезу **«Яркие заголовки email»**.

Почему:
- По RICE это лидер (максимальный приоритет за счет большого охвата и низких трудозатрат).
- По времени реализации гипотеза быстрая (высокая простота = 8), значит подходит для быстрого эксперимента.
- Даже при не максимальном impact даёт хороший ожидаемый продуктовый эффект за короткий срок.

## Задание 2. Метрики эксперимента

- **Ключевая метрика:** конверсия из открытия письма в клик (CTR по открывшим), `clicked / opened`.
- **Смежные метрики:**
  - доля открытий (open rate),
  - абсолютное число кликов,
  - технические метрики качества теста (распределение трафика по группам, стабильность по дням, отсутствие дублей).

Обоснование: цель проекта — увеличить переходы на сайт через рассылки, поэтому именно конверсия в клик отражает целевое действие пользователя.

## Задание 3a. Дата предыдущей акции и рост кликов

In [4]:
open_daily = opened.groupby("date")["user_id"].nunique().rename("opened")
click_daily = clicked.groupby("date")["user_id"].nunique().rename("clicked")
daily = pd.concat([open_daily, click_daily], axis=1).fillna(0)
daily["cr"] = daily["clicked"] / daily["opened"]

# Простой способ найти акцию: дни с выбросами по кликам (IQR) + непрерывный период
q1, q3 = daily["clicked"].quantile([0.25, 0.75])
iqr = q3 - q1
threshold = q3 + 1.5 * iqr
promo_mask = daily["clicked"] > threshold

idx = daily.index[promo_mask]
ranges = []
if len(idx):
    start = idx[0]
    prev = idx[0]
    for d in idx[1:]:
        if (d - prev).days == 1:
            prev = d
        else:
            ranges.append((start, prev))
            start = d
            prev = d
    ranges.append((start, prev))

# Берем самый длинный непрерывный период как акцию
promo_start, promo_end = max(ranges, key=lambda x: (x[1] - x[0]).days)
promo = (daily.index >= promo_start) & (daily.index <= promo_end)

avg_clicks_before = daily.loc[~promo, "clicked"].mean()
avg_clicks_promo = daily.loc[promo, "clicked"].mean()
abs_growth_clicks = avg_clicks_promo - avg_clicks_before
pct_growth_clicks = (avg_clicks_promo / avg_clicks_before - 1) * 100

print("Предполагаемый период акции:", promo_start.date(), "—", promo_end.date())
print(f"Средние клики/день вне акции: {avg_clicks_before:.2f}")
print(f"Средние клики/день в акции:   {avg_clicks_promo:.2f}")
print(f"Рост кликов (абсолютно):      {abs_growth_clicks:.2f} клика/день")
print(f"Рост кликов (относительно):   {pct_growth_clicks:.2f}%")

Предполагаемый период акции: 2023-02-06 — 2023-02-12
Средние клики/день вне акции: 7.79
Средние клики/день в акции:   12.14
Рост кликов (абсолютно):      4.35 клика/день
Рост кликов (относительно):   55.91%


### Вывод по заданию 3a
Наиболее вероятный период прошлой акции: **06.02.2023–12.02.2023**.

Количество кликов в период акции выросло примерно на **4.35 клика/день** (или на **55.9%** относительно обычного периода).

## Задание 3b. Проверка статистической значимости изменения конверсии

- **H0:** конверсия `clicked/opened` в период акции не отличается от обычного периода.
- **H1:** конверсия в период акции отличается (двусторонняя гипотеза).
- **Уровень значимости:** `alpha = 0.05`.
- **Тест:** z-тест двух пропорций.

In [5]:
click_nonpromo = int(daily.loc[~promo, "clicked"].sum())
open_nonpromo = int(daily.loc[~promo, "opened"].sum())
click_promo = int(daily.loc[promo, "clicked"].sum())
open_promo = int(daily.loc[promo, "opened"].sum())

z_3b, p_3b = proportions_ztest(
    [click_nonpromo, click_promo],
    [open_nonpromo, open_promo],
    alternative="two-sided",
)

cr_nonpromo = click_nonpromo / open_nonpromo
cr_promo = click_promo / open_promo

print(f"CR вне акции: {cr_nonpromo:.4%}")
print(f"CR в акции:   {cr_promo:.4%}")
print(f"z-stat:       {z_3b:.3f}")
print(f"p-value:      {p_3b:.6f}")
print("Вывод:", "H0 отвергается" if p_3b < 0.05 else "Нет оснований отвергать H0")

CR вне акции: 9.4937%
CR в акции:   14.4804%
z-stat:       -3.760
p-value:      0.000170
Вывод: H0 отвергается


### Вывод по заданию 3b
Различие статистически значимо (`p < 0.05`): акция действительно повысила конверсию из открытия в клик.

## Задание 4a. Дизайн A/B-теста для гипотезы «яркие заголовки»

1. **Гипотезы:**
   - H0: конверсия в клик в группе B (новые заголовки) равна конверсии в группе A.
   - H1: конверсия в группе B выше, чем в группе A.

2. **Параметры теста:**
   - уровень значимости `alpha = 0.05`,
   - мощность `1 - beta = 0.80`,
   - параметрическая цель — detect uplift, оцененный на исторических данных,
   - для консервативной оценки используем двусторонний расчет размера выборки.

3. **Размер выборки:** считаем по двум пропорциям с baseline и ожидаемым uplift из задания 3.

In [6]:
alpha = 0.05
power = 0.80
traffic_per_day = 240  # ожидаемое число открытий в день суммарно

effect_size = abs(proportion_effectsize(cr_nonpromo, cr_promo))
n_per_group = NormalIndPower().solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    ratio=1.0,
    alternative="two-sided",
)

n_per_group = math.ceil(n_per_group)
n_total = 2 * n_per_group
days_needed = math.ceil(n_total / traffic_per_day)

print("Размер выборки на группу:", n_per_group)
print("Общий размер выборки:", n_total)
print("Длительность теста (дней):", days_needed)

Размер выборки на группу: 660
Общий размер выборки: 1320
Длительность теста (дней): 6


## Задание 4b. Вопросы

1. Вероятность ошибки 1-го рода задает: **уровень статистической значимости**.
2. За вероятность ошибки 2-го рода отвечает: **мощность статистического теста** (через `beta = 1 - power`).
3. При уменьшении вероятности ошибки 1-го рода (`alpha`) требуемый размер выборки: **увеличится**.

## Задание 5. Мониторинг первых 4 дней: проверка корректности эксперимента

In [7]:
monitoring.head()

,date,group,user_id,converted
0,2023-03-01,control,106085,1
1,2023-03-01,control,106086,1
2,2023-03-01,control,106087,1
3,2023-03-01,control,106088,1
4,2023-03-01,control,106089,1


In [8]:
# 1) Проверка целостности
print("Пропуски:\n", monitoring.isna().sum(), "\n")
print("Дубликаты user_id:", monitoring.duplicated("user_id").sum(), "\n")

# 2) Проверка SRM (перекос трафика)
group_counts = monitoring["group"].value_counts().sort_index()
exp = np.array([group_counts.sum() / 2, group_counts.sum() / 2])
chi2 = ((group_counts.values - exp) ** 2 / exp).sum()
p_srm = 1 - stats.chi2.cdf(chi2, df=1)

print("Распределение по группам:\n", group_counts, "\n")
print(f"SRM p-value: {p_srm:.3e}\n")

# 3) Динамика по дням
by_day_n = monitoring.groupby(["date", "group"])["user_id"].nunique().unstack(fill_value=0)
by_day_cr = monitoring.groupby(["date", "group"])["converted"].mean().unstack()

print("Размер групп по дням:\n", by_day_n, "\n")
print("Конверсия по дням:\n", by_day_cr)

Пропуски:
 date         0
group        0
user_id      0
converted    0
dtype: int64 

Дубликаты user_id: 0 

Распределение по группам:
 group
control      603
treatment    363
Name: count, dtype: int64 

SRM p-value: 1.144e-14

Размер групп по дням:
 group       control  treatment
date                          
2023-03-01      119        122
2023-03-02      127        119
2023-03-03      126        111
2023-03-04      231         11 

Конверсия по дням:
 group        control  treatment
date                           
2023-03-01  0.100840   0.114754
2023-03-02  0.094488   0.126050
2023-03-03  0.095238   0.000000
2023-03-04  0.095238   0.000000


### Вывод по заданию 5
В первых 4 днях есть выраженные аномалии:
- Сильный SRM (группы распределены не 50/50, p-value крайне мал).
- На 4-й день почти весь трафик уходит в control (231 vs 11).
- В treatment в последние 2 дня нулевая конверсия.

Это признаки проблем рандомизации/треккинга, поэтому результатам первых 4 дней доверять нельзя без расследования.

Способы выявления аномалий:
- ежедневный SRM-чек (chi-square / binomial),
- контроль размера групп по дням и по источникам,
- контроль аномальных скачков CR по группам,
- проверки на дубли, пропуски, задержки событий и рассинхрон timestamp.

## Задание 6. Итоги A/B-теста на финальных данных

In [9]:
agg = results.groupby("group")["converted"].agg(["sum", "count", "mean"])
agg

,sum,count,mean
group,,,
control,61,613,0.099511
treatment,75,592,0.126689


In [10]:
control_success = int(agg.loc["control", "sum"])
control_n = int(agg.loc["control", "count"])
treat_success = int(agg.loc["treatment", "sum"])
treat_n = int(agg.loc["treatment", "count"])

z_res, p_res = proportions_ztest(
    [control_success, treat_success],
    [control_n, treat_n],
    alternative="two-sided",
)

print(f"CR control:   {control_success/control_n:.3%}")
print(f"CR treatment: {treat_success/treat_n:.3%}")
print(f"z-stat: {z_res:.3f}")
print(f"p-value: {p_res:.6f}")

for alpha_level in [0.001, 0.01, 0.05, 0.15]:
    decision = "отвергаем H0" if p_res < alpha_level else "не отвергаем H0"
    print(f"alpha={alpha_level}: {decision}")

CR control:   9.951%
CR treatment: 12.669%
z-stat: -1.491
p-value: 0.136074
alpha=0.001: не отвергаем H0
alpha=0.01: не отвергаем H0
alpha=0.05: не отвергаем H0
alpha=0.15: отвергаем H0


### Вывод по заданию 6
- При `alpha = 0.001, 0.01, 0.05`: статистически значимых различий нет.
- При `alpha = 0.15`: различие становится значимым.

Практический вывод: на стандартных порогах значимости (0.05 и ниже) убедительных оснований выкатывать вариант B нет.

## Задание 7. Множественная проверка гипотез (5 гипотез)

In [11]:
np.random.seed(42)
alpha = 0.05
n_per_group = 200
n_tests = 5

raw_pvals = []
for _ in range(n_tests):
    a = np.random.normal(loc=0.0, scale=1.0, size=n_per_group)
    b = np.random.normal(loc=0.0, scale=1.0, size=n_per_group)
    p = stats.ttest_ind(a, b, equal_var=False).pvalue
    raw_pvals.append(p)

df_multi = pd.DataFrame({
    "test_id": [f"H{i+1}" for i in range(n_tests)],
    "p_value_raw": raw_pvals,
})

# Поправка Бонферрони
df_multi["alpha_bonf"] = alpha / n_tests
df_multi["reject_raw_alpha_0_05"] = df_multi["p_value_raw"] < alpha
df_multi["reject_bonf"] = df_multi["p_value_raw"] < (alpha / n_tests)
df_multi["p_value_bonf_adjusted"] = (df_multi["p_value_raw"] * n_tests).clip(upper=1.0)

df_multi

,test_id,p_value_raw,alpha_bonf,reject_raw_alpha_0_05,reject_bonf,p_value_bonf_adjusted
0,H1,0.187608,0.01,False,False,0.938039
1,H2,0.347920,0.01,False,False,1.000000
2,H3,0.956843,0.01,False,False,1.000000
3,H4,0.445174,0.01,False,False,1.000000
4,H5,0.326907,0.01,False,False,1.000000


### Вывод по заданию 7
После поправки Бонферрони критерий становится строже (`alpha/5 = 0.01`), поэтому число ложноположительных результатов уменьшается.

Это и есть цель множественной коррекции: контролировать суммарную вероятность ошибки 1-го рода при множественных проверках.